## The Vault CLI and its tests

A small development repository still needs credentials. An API client needs a
token; an image uploader needs access to an object store. Keeping those values in
project files makes it easy for a repository read, a diagnostic command, or an
accidental commit to expose them.

Atlas puts local secret values in the system keyring and provides four commands
for working with them. On the current development platform, macOS, the intended
backend is Keychain. The project calls this small command group the Vault.

This walkthrough in [Building Atlas](../collections/building-atlas.qmd) covers
the CLI and its tests, then its audit history, retention, and coordination
between simultaneous processes. The CLI excerpts focus on command behavior;
the audit section explains the wrappers in the current source.

**1. Start with a command contract.** The Vault needs a way to store, retrieve,
remove, and check credentials. Each operation has a defined input and output:

| Command | Input | Result |
| --- | --- | --- |
| `at vault set NAME` | Hidden prompt, or explicit `--stdin` | Store or replace a value. |
| `at vault get NAME` | Secret name | Copy to the clipboard in a terminal, or write to stdout when captured. |
| `at vault rm NAME` | Confirmation, or explicit `--yes` | Remove the local keyring entry. |
| `at vault check` | Names from `.env.example` | Report which entries exist, without displaying their values. |

The distinction between a secret's name and value matters throughout the design.
`API_TOKEN` is a name that can appear in a manifest and a diagnostic message.
The credential stored under that name should not appear there.

The normal data path is short:

```{mermaid}
%%| label: fig-vault-cli-path
%%| fig-cap: "A Vault command names a credential. The keyring backend owns its stored value."
flowchart TD
    User["Terminal or script"] --> Root["at: root Typer app"]
    Root --> Vault["vault: command group"]
    Vault --> Command["set / get / rm / check"]
    Manifest[".env.example: names only"] --> Command
    Command --> Backend["Python keyring backend"]
    Backend --> Keychain["macOS Keychain"]
```

The diagram describes the default local setup. Python keyring selects its
backend, so backend configuration remains part of the environment. The CLI
rejects the disabled and unavailable backends; it does not certify every
third-party backend as secure.

**2. Connect the executable to a Typer command group.** The executable entry in
`pyproject.toml` points to the root app's `main` function:

```toml
[project.scripts]
at = "atlas.cli:main"
```

The root app mounts a second Typer app under the name `vault`:

```python
import typer

from atlas.vault import app as vault_app

app = typer.Typer(
    no_args_is_help=True,
    help="Atlas tools.",
    pretty_exceptions_show_locals=False,
)
app.add_typer(vault_app, name="vault")


def main() -> None:
    app()
```

Disabling local-variable display in pretty tracebacks reduces the chance that a
secret held in a local variable appears in a diagnostic. Expected keyring
failures get their own shorter error path later in the implementation.

The Vault app registers commands with decorators such as
`@app.command("get")`. Their Python function names can stay descriptive while
the public commands stay short. After installing the project dependencies, the
entry point is available through `uv run`:

```sh
uv sync
uv run at vault --help
```

**3. Give all keyring operations one namespace and one error policy.** Keyring
looks up a value using a service name and a username. Atlas uses the constant
`atlas.vault` as the service and the secret's name as the username.

Conceptually, `API_TOKEN` is stored under this pair:

```python
("atlas.vault", "API_TOKEN")
```

That namespace is shared across projects for the current OS user. Changing the
working directory does not create a separate Vault. Project-specific names or
credentials therefore need deliberate naming and permission choices.

The wrapper around keyring calls supplies the service name and sanitizes
expected failures:

```python
def _call_keyring[T](function: Callable[..., T], *args: str) -> T:
    try:
        return function(SERVICE_NAME, *args)
    except Exception as error:
        _exit(f"Keyring operation failed ({type(error).__name__}).")
```

This excerpt uses the generic-function syntax supported by the project's Python
3.13 requirement. `T` preserves the wrapped function's return type: a lookup can
return a string or `None`, while a write can return `None`.

The error message includes the exception's type, rather than its full text.
A backend exception could include a credential in its message. There is no need
to reproduce that message to explain that the operation failed.

**4. Make secret input explicit.** The interactive command prompts for a hidden
value and asks for confirmation:

```sh
uv run at vault set API_TOKEN
```

The prompt configuration in `set_secret()` is:

```python
with warnings.catch_warnings():
    warnings.simplefilter("error", GetPassWarning)
    value = typer.prompt(
        "Secret", hide_input=True, confirmation_prompt=True, err=True
    )
```

The code first requires `sys.stdin.isatty()` to be true. A **TTY** is a terminal
interface; `isatty()` tells the command whether the stream is attached to one.
Captured input in a test runner or a pipe normally does not satisfy that check.

The warning filter treats `GetPassWarning` as an exception. Python uses this
warning when its password-input function cannot guarantee echo-free input. The
Vault catches it and directs the caller to the explicit stdin path.
[Python's password-input documentation](https://docs.python.org/3.13/library/getpass.html).

A script supplies `--stdin` deliberately. For example, an existing credential
file can feed the command without putting the value in its arguments:

```sh
uv run at vault set API_TOKEN --stdin < /path/to/existing-secret
```

The implementation reads stdin and removes one trailing line ending:

```python
value = sys.stdin.read()
value = value.removesuffix("\n").removesuffix("\r")
```

It does not call `strip()`, which would also remove potentially meaningful
spaces. Empty input is rejected. Other whitespace is preserved apart from that
single trailing line ending.

**5. Separate data output from status messages.** Retrieval has two behaviors,
selected by `sys.stdout.isatty()`:

```python
if sys.stdout.isatty():
    _copy_to_clipboard(value)
    typer.echo(f"Copied {name} to clipboard.", err=True)
else:
    sys.stdout.write(value)
```

In a terminal, `pbcopy` receives the value through its stdin and a status message
goes to stderr. When stdout is captured or redirected, stdout contains exactly
the stored value, with no added newline.

This makes the command usable in scripts, but it is also an intentional release
of the credential. A tool that captures stdout can receive the real value. The
Vault does not promise to hide a retrieved credential from the caller.

Exit codes are another part of the command contract:

| Code | Meaning in the current CLI |
| --- | --- |
| `0` | Success, including a user declining deletion. |
| `1` | Missing secret, missing manifest, or an uncaught ordinary failure. |
| `2` | Invalid manifest, invalid input, or a handled keyring/clipboard error. |

Deletion requires confirmation in a terminal. In scripts, `--yes` or `-y` supplies
that confirmation explicitly. Removing a keyring entry removes the local copy;
revoking the credential at its provider is a separate operation.

**6. Keep the manifest about names.** The repository-root `.env.example` declares
expected entries using empty assignments:

```dotenv
# Required secrets
API_TOKEN=
DATABASE_URL=
```

`check` skips blank lines and full-line comments, requires a nonempty name and an
empty value, then looks up the declared names. It reports `[ok]` or `[no]` and a
count. It does not modify the manifest or display stored values.

This is a small parser rather than a shell interpreter. Its path comes from
`ROOT_PATH`, not the caller's current directory. A missing manifest raises
`FileNotFoundError`; a manifest with no names reports `No secrets declared.`
and succeeds.

A mistaken line such as `API_TOKEN=private value` is rejected without printing
the supplied value. Error handling needs that property too: rejecting an input
does not justify echoing its contents.

**7. Replace Keychain in tests.** Tests should be able to create and delete
credentials freely without changing the developer's real keyring. The test
suite uses an in-memory implementation of the keyring interface:

```python
from keyring.backend import KeyringBackend
from keyring.errors import PasswordDeleteError


class LocalKeyring(KeyringBackend):
    priority = 1

    def __init__(self) -> None:
        self.values: dict[tuple[str, str], str] = {}

    def get_password(self, service: str, username: str) -> str | None:
        return self.values.get((service, username))

    def set_password(self, service: str, username: str, password: str) -> None:
        self.values[(service, username)] = password

    def delete_password(self, service: str, username: str) -> None:
        try:
            del self.values[(service, username)]
        except KeyError as error:
            raise PasswordDeleteError from error
```

Each test gets a fresh instance. The dictionary key matches the service/name
pair used by the real library, so tests exercise the namespace and the complete
store/retrieve/remove behavior.

Fixtures connect this substitute to the application:

```python
from unittest.mock import Mock

import pytest
from typer.testing import CliRunner


@pytest.fixture
def runner() -> CliRunner:
    return CliRunner()


@pytest.fixture
def backend() -> LocalKeyring:
    return LocalKeyring()


@pytest.fixture(autouse=True)
def get_backend(monkeypatch, backend) -> Mock:
    replacement = Mock(return_value=backend)
    monkeypatch.setattr("atlas.vault.keyring.get_keyring", replacement)
    return replacement


@pytest.fixture
def manifest(tmp_path, monkeypatch):
    monkeypatch.setattr("atlas.vault.ROOT_PATH", tmp_path)
    return tmp_path / ".env.example"


@pytest.fixture(autouse=True)
def audit_log(tmp_path, monkeypatch):
    monkeypatch.setattr("atlas.vault_audit.ROOT_PATH", tmp_path)
    monkeypatch.delenv("ATLAS_VAULT_SESSION_ID", raising=False)
    return tmp_path / ".atlas" / "logs" / "vault.jsonl"
```

`autouse=True` applies the backend replacement even when a test does not name
that fixture. `tmp_path` supplies a temporary directory for a test's manifest.
`monkeypatch` restores the replaced attributes after the test.
[pytest's monkeypatch documentation](https://docs.pytest.org/en/stable/how-to/monkeypatch.html).

The last fixture isolates the audit log described below, which is already
present in the current source. It patches `atlas.vault_audit.ROOT_PATH` separately
and clears `ATLAS_VAULT_SESSION_ID`.
Patching the name in one module does not change a constant already imported by
another module.

```{mermaid}
%%| label: fig-vault-cli-test-path
%%| fig-cap: "CLI tests retain argument parsing and application behavior while replacing external state."
flowchart TD
    Test["pytest test"] --> Runner["CliRunner.invoke(app, arguments)"]
    Runner --> CLI["Real Typer command"]
    CLI --> Fake["Fresh LocalKeyring dictionary"]
    CLI --> Manifest["Manifest under tmp_path"]
    CLI --> Result["Exit code, stdout, stderr"]
    Result --> Assertions["Check behavior and secret exclusion"]
```

**8. Test the public CLI, including its output streams.** `CliRunner` invokes the
Typer app with an argument list and optional simulated input. This exercises
command registration and parsing as well as the command body.
[Typer's testing guide](https://typer.tiangolo.com/tutorial/testing/).

The following test is a compact version of the lifecycle covered in
`tests/test_vault.py`. It uses the fixtures above and imports the actual root
app and namespace:

```python
from atlas.cli import app
from atlas.vault import SERVICE_NAME


def test_secret_lifecycle(runner, backend, manifest):
    contents = "API_TOKEN=\n"
    manifest.write_text(contents)

    stored = runner.invoke(
        app, ["vault", "set", "API_TOKEN", "--stdin"],
        input="dummy credential\n",
    )
    assert stored.exit_code == 0
    assert stored.stdout == ""
    assert "dummy credential" not in stored.output
    assert backend.values == {(SERVICE_NAME, "API_TOKEN"): "dummy credential"}

    retrieved = runner.invoke(app, ["vault", "get", "API_TOKEN"])
    assert retrieved.exit_code == 0
    assert retrieved.stdout == "dummy credential"

    checked = runner.invoke(app, ["vault", "check"])
    assert checked.exit_code == 0
    assert "[ok]  API_TOKEN" in checked.stdout
    assert "dummy credential" not in checked.output

    removed = runner.invoke(app, ["vault", "rm", "API_TOKEN", "--yes"])
    assert removed.exit_code == 0
    assert backend.values == {}
    assert manifest.read_text() == contents
```

The retrieval assertion intentionally expects the credential in stdout: captured
output takes the nonterminal branch. The storage and check assertions expect it
to be absent. A blanket assertion that a secret can never appear in output would
contradict `get`'s public behavior.

Checking `result.stdout` separately from `result.stderr` also catches a subtle
regression: a friendly status message written to stdout could corrupt data sent
to another program. In this suite, `result.output` is useful for checking the
combined visible output for accidental disclosure.

**9. Exercise failures as contracts.** The error tests cover behavior the happy
path cannot establish:

| Test case | Property being checked |
| --- | --- |
| Empty or noninteractive input without `--stdin` | Invalid input cannot create a secret. |
| Missing secret | Exit code `1`, with no value on stdout. |
| Invalid manifest | Failure occurs before keyring lookup, without echoing its contents. |
| Missing manifest | The current `FileNotFoundError` behavior is explicit. |
| Disabled or unavailable backend | Failure is reported rather than silently ignored. |
| Unconfirmed noninteractive deletion | The stored entry survives. |
| Backend exception containing a dummy credential | The visible error omits that credential. |

For the last case, the test injects a deliberately unsafe backend message:

```python
def test_backend_error_is_sanitized(runner, backend, monkeypatch):
    monkeypatch.setattr(
        backend,
        "set_password",
        Mock(side_effect=RuntimeError("dummy credential")),
    )

    result = runner.invoke(
        app, ["vault", "set", "TOKEN", "--stdin"],
        input="dummy credential",
    )

    assert result.exit_code == 2
    assert "RuntimeError" in result.output
    assert "dummy credential" not in result.output
```

The message still identifies the failure category. The test establishes that
the backend's arbitrary exception text does not reach the user-facing error.

Interactive branches need a different setup because `CliRunner` is not a real
terminal. The existing test replaces the Vault module's `sys` reference with a
small object whose stdin reports `isatty() == True`, then supplies a mismatched
confirmation followed by matching inputs. It separately declines and accepts
deletion. Another test makes `typer.prompt` raise `GetPassWarning` and verifies
that nothing is stored.

Those tests establish control flow. They do not demonstrate actual terminal echo
behavior, a Keychain authorization dialog, or the contents of the system
clipboard. A manual smoke check with a disposable credential remains useful
when changing those integrations.

**10. Run the checks and keep their scope clear.** From the repository root,
the CLI tests and the full suite can be run separately:

```sh
uv run pytest tests/test_vault.py -q
uv run pytest --cov=atlas --cov-branch --cov-report=term-missing
```

`make doctor` additionally runs `at vault check` against the real local manifest
and keyring. That makes it an environment check as well as a test command; it is
different from the isolated pytest run.

The Vault reduces accidental exposure from project files and provides a
consistent interface for local secrets. It does not restrict a credential's
remote permissions. A B2 application key that can delete objects still has that
authority after retrieval. Keyring also documents that scripts using the same
Python executable can access its macOS entries without another password prompt
under the described access settings.
[Keyring's security considerations](https://keyring.readthedocs.io/en/stable/#security-considerations).

The next section makes supported command use visible through an audit history.


## Audit history and concurrency

The Vault also keeps a history of supported command use: which operation ran,
which secret names it targeted, and whether it succeeded.

That history is useful when a person or an agent invokes the CLI. It helps
answer questions such as whether a missing credential was recently removed or
which development session retrieved a particular name. The history needs to
exclude the credential values themselves.

This section follows `src/atlas/vault_audit.py` and its integration with
`src/atlas/vault.py`. It assumes macOS and explains file locking from the beginning.

**1. Define what an event records.** The command supplies an operation, a list of
targeted names, an outcome, and an exit code. The writer adds a UTC timestamp,
process ID, and optional session label. A fictional record looks like this,
expanded onto several lines for readability:

```json
{
  "timestamp": "2026-10-28T06:00:00+00:00",
  "operation": "get",
  "secret_names": ["B2_DEV_KEY"],
  "outcome": "success",
  "exit_code": 0,
  "pid": 12345,
  "session_id": "experiment-1"
}
```

The real file, `.atlas/logs/vault.jsonl`, has one JSON object per line. It is easy
to inspect with ordinary file tools and can be read one record at a time.

The event object deliberately has no field for a value, stdin, a captured command
output, or an exception message:

```python
from dataclasses import dataclass, field


@dataclass
class AuditEvent:
    operation: str
    secret_names: list[str] = field(default_factory=list)
    outcome: str = "success"
    exit_code: int = 0
```

`default_factory=list` gives every event its own list. That prevents an event's
names from accidentally becoming shared mutable state.

`secret_names` describes the command's targets. For `check`, it contains all the
valid manifest names even if an early backend error prevents every lookup from
completing. It is not a sequence of individual backend-access receipts.

**2. Wrap commands in a context manager.** The command integration is small:

```python
with audit("get", name):
    value = _call_keyring(_get_backend().get_password, name)
    # Existing missing-secret and output handling follows here.
```

The context first prunes old records, creates an event, and yields control to the
command. When the command finishes, it classifies the result and saves the event.
The complete outcome-handling portion is:

```python
@contextmanager
def audit(operation: str, name: str | None = None) -> Generator[AuditEvent, None, None]:
    _update_log()
    event = AuditEvent(operation, [name] if name is not None else [])
    try:
        yield event
    except typer.Exit as error:
        event.exit_code = error.exit_code
        if error.exit_code:
            event.outcome = "failure"
        raise
    except (typer.Abort, KeyboardInterrupt):
        event.outcome = "cancelled"
        event.exit_code = 1
        raise
    except Exception:
        event.outcome = "failure"
        event.exit_code = 1
        raise
    finally:
        _update_log(event)
```

`@contextmanager` turns this generator into something that works with `with`.
Execution pauses at `yield event` while the command body runs. If the body raises
an exception, Python brings it back into the generator at that handoff point.
The handlers update the event and re-raise the exception, preserving the CLI's
normal error behavior. The `finally` block saves the outcome on ordinary exits
and handled exception paths.
[Python's context manager documentation](https://docs.python.org/3.13/library/contextlib.html#contextlib.contextmanager).

The `Generator` annotation describes the decorated function's generator: it
yields an `AuditEvent`, takes no meaningful sent-in value, and returns no final
value. The lock context used later has the same structure but yields `None`.

Typer can use `Exit(0)` for a successful command. The handler therefore checks the
exit code instead of treating every `Exit` as a failure. Declining deletion is
another special case: `rm` explicitly changes the event to `cancelled` and
returns normally, leaving exit code `0`.

The whole command path is:

```{mermaid}
%%| label: fig-vault-audit-lifecycle
%%| fig-cap: "The command runs between two short log updates. The file lock is released before the command body begins."
flowchart TD
    Start["Enter audit context"] --> Prune["Lock, prune old records, publish log, unlock"]
    Prune --> Event["Create AuditEvent"]
    Event --> Run["Run Vault command"]
    Run --> Outcome["Keep success or record failure / cancellation"]
    Outcome --> Save["Lock, prune, add outcome record, publish log, unlock"]
    Save --> Return["Return or propagate the original exception"]
```

There is no start record written before the command. A process terminated before
its finalizer runs can leave no outcome record. This is a local command history
with known coverage, rather than a complete record of all credential access.

**3. See why two writers need coordination.** A process is a running program.
Two terminals or two agent sessions can start separate Vault processes, each
with its own memory and its own copy of the log data it has read.

Suppose both processes read the same existing history. Process A adds event A
and saves it. Process B still has its older copy, so it adds event B and saves
that version. Event A disappears from the final file.

This is a **lost update**. The problem spans the whole read/modify/write sequence.
Protecting only the final write would still allow both writers to prepare
different replacements from the same old history.

The implementation makes that whole sequence exclusive. Each writer must
acquire the same OS lock before reading the log, and release it after publishing
the replacement.

```{mermaid}
%%| label: fig-vault-lock-sequence
%%| fig-cap: "An exclusive lock makes the second Vault process wait until the first has published its update."
sequenceDiagram
    participant A as Vault process A
    participant OS as macOS
    participant B as Vault process B
    A->>OS: Acquire exclusive lock
    OS-->>A: Granted
    B->>OS: Acquire the same lock
    Note over B: Waits inside flock
    Note over A: Read, prune, add event, replace log
    A->>OS: Release lock
    OS-->>B: Granted
    Note over B: Read the latest log and update it
    B->>OS: Release lock
```

The second process reads after it has acquired the lock, so its starting history
includes the first process's completed update.

**4. Acquire and release the macOS file lock.** The helper is small enough to
read in full:

```python
@contextmanager
def _lock(path: Path) -> Generator[None, None, None]:
    descriptor = os.open(path, os.O_CREAT | os.O_RDWR, 0o600)
    with os.fdopen(descriptor, "a+b") as lock:
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX)
        try:
            yield
        finally:
            fcntl.flock(lock.fileno(), fcntl.LOCK_UN)
```

`os.open()` creates the lock file if necessary and returns a **file descriptor**,
a small integer identifying an open file to the OS. `os.fdopen()` wraps that
descriptor in a Python file object. The `0o600` mode requests read/write access
for the owner and no permissions for the group or other ordinary users when the
file is created.

`fcntl.flock(..., LOCK_EX)` asks macOS for an exclusive lock. A competing request
for the same locked file normally waits until it can proceed. `LOCK_UN` releases
the lock. These calls coordinate cooperating processes; they do not prevent an
uncooperative program from modifying the file without requesting the lock.
[Apple's file-locking documentation](https://developer.apple.com/library/archive/documentation/System/Conceptual/ManPages_iPhoneOS/man2/flock.2.html).

The helper uses `finally` so an ordinary error inside the `with` block still
releases the lock. The outer `with` closes the Python file object too.

The lock file is `.atlas/logs/.vault.lock`. Its existence does not mean a process
currently holds the lock. macOS tracks the live lock state; the file stays in
place between commands.

Only log updates acquire this lock. The process releases it before reading a
credential, waiting for hidden input, or asking for deletion confirmation.
A person pausing at a prompt therefore does not hold up other log updates.

**5. Apply a rolling 30-day retention window.** Once the writer holds the lock,
it reads the current time in UTC and calculates the oldest timestamp it will
keep:

```python
now = datetime.now(UTC)
cutoff = now - RETENTION  # RETENTION = timedelta(days=30)
```

For each JSON line, it parses the timestamp and retains the record only when
this condition holds:

```python
timestamp.tzinfo is not None and cutoff <= timestamp <= now
```

The cutoff is inclusive. A record exactly 30 days old is retained at that
instant; an older record is removed. A timezone is required so the comparison
has a defined meaning. Future records are discarded too.

Malformed JSON, missing timestamps, and invalid timestamp values cannot satisfy
the retention rule, so cleanup drops those records. The log is rewritten with
only the retained history and, on the completion path, the new event.

This is cleanup on use. Every command prunes at entry and completion, but no
background service runs while the Vault is idle. Filesystem failures can also
prevent cleanup and generate a warning. The implementation does not promise
deletion at the exact expiration instant while no command is running.

**6. Publish a complete replacement file.** The writer does not truncate the
current log and then rebuild it in place. It writes the updated history into a
temporary file in the same directory:

```python
descriptor, temporary = tempfile.mkstemp(
    prefix=".vault-", suffix=".tmp", dir=directory
)
try:
    with os.fdopen(descriptor, "w", encoding="utf-8") as destination:
        for record in records:
            destination.write(json.dumps(record) + "\n")
    os.replace(temporary, path)
finally:
    Path(temporary).unlink(missing_ok=True)
```

`mkstemp()` creates a distinct temporary file with owner-only access on this
platform. Placing it beside the target keeps the replacement within the same
filesystem. The final `os.replace()` publishes the completed file atomically:
readers can open the old version or the completed replacement, rather than a
partially rewritten target.
[Temporary-file creation](https://docs.python.org/3.13/library/tempfile.html#tempfile.mkstemp),
[atomic replacement](https://docs.python.org/3.13/library/os.html#os.replace).

```{mermaid}
%%| label: fig-vault-atomic-replacement
%%| fig-cap: "The existing log remains available while the replacement is prepared. A separate lock file coordinates writers across replacements."
flowchart TD
    Lock["Hold .vault.lock"] --> Read["Read vault.jsonl"]
    Read --> Filter["Keep recent records and add the new event"]
    Filter --> Temp["Write a complete .vault-unique.tmp file"]
    Temp --> Replace["os.replace: publish as vault.jsonl"]
    Replace --> Release["Release .vault.lock"]
```

The exclusive lock prevents lost updates between writers. Atomic replacement
protects the visible file from a partial rewrite. These are separate properties.
Atomic replacement alone also does not promise that the newest event survives
an abrupt power loss; this implementation does not add a disk-sync protocol.

A separate lock file is important here. Locking the log itself and then
replacing it would leave the lock associated with the old file, while another
process could open the replacement. Keeping `.vault.lock` stable gives all
supported writers a common object to coordinate through. It should not be
deleted as routine cleanup.

**7. Keep runtime files local and failures visible.** All runtime files belong
under one directory:

```text
.atlas/logs/
  vault.jsonl
  .vault.lock
  .vault-*.tmp
```

The repository's `.gitignore` contains `/.atlas/logs/`, so it covers the log,
temporary files, and lock file together. The log directory is created with
mode `0o700`: its owner can list, modify, and traverse it. The new log files
have mode `0o600`. These permissions limit other ordinary OS users; an agent
running as the owner shares that user's access.

The log writer catches filesystem errors and emits a short warning to stderr:

```text
Warning: Could not update Vault audit log (PermissionError).
```

It includes the exception type and omits the raw exception message. A failure to
write this diagnostic history does not change the Vault command's result or
mix a warning into its stdout data.

A session label can be supplied explicitly:

```sh
ATLAS_VAULT_SESSION_ID=experiment-1 uv run at vault check
```

The writer stores at most 256 characters; an absent or empty label becomes
`null`. This is self-reported context. The process ID identifies the running
process, but neither field authenticates an agent or establishes human approval.
There are no signatures in this implementation.

**8. Test retention, privacy, and real competing processes.** The tests use
temporary directories, so they do not touch the developer's actual audit file.
The CLI fixture patches `atlas.vault_audit.ROOT_PATH` and clears the optional
session environment variable.

The retention test fixes the clock instead of waiting 30 days. Its seeded file
contains records just before the cutoff, exactly at the cutoff, at the current
time, in the future, and with invalid timestamps. It verifies cleanup inside
the audit block, proving that pruning happens before the command body. It then
advances the clock by 31 days and checks that the old history disappears.

Privacy tests exercise the actual CLI lifecycle and an exception whose message
contains a dummy credential. They verify that values do not enter the saved
records. Another test puts newlines and JSON-looking text in the name and session
label. JSON encoding keeps that event on one physical line rather than creating
fake entries.

The concurrency test launches four separate Python processes. Each creates
three events against the same temporary log directory. A thread pool starts
those subprocesses concurrently; the competing log writers are OS processes,
not merely tasks sharing one Python interpreter.

The assertions check every expected event, temporary-file cleanup, and the log's
permissions:

```python
records = [json.loads(line) for line in audit_log.read_text().splitlines()]
assert len(records) == 12
assert {record["secret_names"][0] for record in records} == {
    f"worker-{worker}-{index}"
    for worker in range(4)
    for index in range(3)
}
assert not list(audit_log.parent.glob("*.tmp"))
assert S_IMODE(audit_log.stat().st_mode) == 0o600
```

Finally, a CLI test places an ordinary file where the `.atlas` directory would
need to be. The resulting write failure must produce a warning on stderr while
`get` still returns the exact dummy credential on stdout. That checks the
chosen failure policy without relying on permission bits behaving identically
under every test runner's privileges.

Run the audit tests alone, or include them in the full suite:

```sh
uv run pytest tests/test_vault_audit.py -q
uv run pytest --cov=atlas --cov-branch --cov-report=term-missing
```

**9. Read the history within its coverage.** The recorded outcomes distinguish
normal completion, errors, and cancellation:

| Situation | Outcome | Exit code |
| --- | --- | --- |
| Normal command completion | `success` | `0` |
| Missing secret or missing entries in `check` | `failure` | `1` |
| Handled input or backend error | `failure` | `2` |
| User declines deletion | `cancelled` | `0` |
| Prompt abort or Ctrl+C | `cancelled` | `1` |

Only calls that enter the Vault wrappers receive these records. Calling the
keyring backend directly bypasses the log. A process running as the same OS user
can alter or delete the history, and the advisory lock does not stop it from
doing so. The log also records credential retrieval, not later requests made to
an API with that credential.

Provider permissions remain the control on those later actions. The audit log
adds a readable account of supported CLI use, with explicit retention and
concurrency behavior, while leaving credential authority with the keyring and
the external service.
